In [1]:
!nvidia-smi

Tue Apr  7 17:48:12 2026
+--------------------------------------------------------------------------------------------+
| SMI N/A                      Driver Version: N/A                      CUDA Version: N/A    |
+-----------------------------------------------+----------------------+---------------------+
| GPU      Name                    Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC|
| Fan      Temp       Perf         Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M.|
|                                               |                      |               MIG M.|
|===============================================+======================+=====================|
|   0      B1.gpu.xlarge                   On   |   N/A            Off |                   0 |
| N/A      35C        P0              N/A / N/A |      0MiB / 24258MiB |      0%     Default |
|                                               |                      |                 N/A |
+------------------------

## 5.6.1 计算设备

In [2]:
import torch
from torch import nn

torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [3]:
# 查询可用gpu的数量
torch.cuda.device_count()

2

In [4]:
def try_gpu(i=0):
    """如果存在, 则返回gpu(i), 否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():
    """返回所有可用的GPU, 如果没有GPU, 则返回[cpu(),]"""
    devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0), device(type='cuda', index=1)])

## 5.6.2 张量与GPU

In [5]:
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

### 存储在GPU上

In [6]:
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [7]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y

tensor([[0.9449, 0.3883, 0.0530],
        [0.2382, 0.3571, 0.6329]], device='cuda:1')

### 复制

In [8]:
Z = X.cuda(1)
print(X)
print(Z)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:1')


In [9]:
Y + Z

tensor([[1.9449, 1.3883, 1.0530],
        [1.2382, 1.3571, 1.6329]], device='cuda:1')

In [10]:
Z.cuda(1) is Z

True

## 5.6.3 神经网络与GPU

In [11]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

In [12]:
net(X)

tensor([[1.4260],
        [1.4260]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [13]:
net[0].weight.data.device

device(type='cuda', index=0)